# Phase 3 · Rhetorical Role Prediction (CL + IT) from the Phase 1 legal LMs

One parameterised notebook, one run per (Phase 1 architecture × adaptation method). Every run starts from a **Phase 1 snapshot** (no external pretrained weights/APIs), adds a sentence-level head, trains on CL+IT train, selects the best epoch on CL+IT dev macro-F1, and writes a Codabench-ready `submission.zip` (`prediction.json` at the zip root).

**Model.** Phase 1 causal decoder → per-sentence `[mean-pool ‖ last-token]` over the sentence's tokens (a `<eos>` separator closes each sentence) → MLP (+ learned CL/IT domain embedding) → optional **BiGRU across the sentences of the window** → 13-way classifier.
**Context.** `CONTEXT='sentence'` encodes each sentence alone (baseline). `CONTEXT='window'` packs ≤12 core sentences plus 3 left / 3 right context sentences into one ≤512-token sequence: the causal decoder supplies left context inside the backbone, and the BiGRU supplies right context. Loss and predictions use core sentences only, so each sentence is predicted exactly once, in order.

**Methods** (`METHOD`): `full`, `partial` (top blocks), `lora`, `rslora`, `lorma` (LoRMA+, the Phase 2 multiplicative adapter), `ia3`, `prefix`, `kd` (6-layer student cut from a Phase 1 model, distilled from a fine-tuned 12-layer teacher with CE + temperature-KL).

### Suggested run plan (each Phase 1 architecture used at least once)
| # | PRESET | METHOD | role |
|---|---|---|---|
| 1 | MHA-prenorm | full | baseline + **KD teacher** |
| 2 | GQA-prenorm | lorma | **compulsory LoRMA** |
| 3 | MLA-prenorm | lora (or rslora) | low-rank |
| 4 | MHA-postnorm | ia3 (or prefix) | PEFT, different mechanism |
| 5 | GQA-prenorm | kd (teacher = run 1) | **mandatory KD**, 6-layer student |
| 6+ | any | same method, `CONTEXT='sentence'` / `CLASS_WEIGHT` variants | context & balancing ablations |

Put `rr_lib.py` next to `phase2_lner_multimethod_bundle.zip` in `MyDrive/legal-llm-course/` first. LLM-assistance note: this code was drafted with an LLM; declare it in the mandatory LLM Declaration. Cite: LoRA (Hu 2021), rsLoRA (Kalajdzievski 2023), IA³ (Liu 2022), Prefix-Tuning (Li & Liang 2021), LoRMA (Bihany 2025), Hinton 2015 (KD).

In [ ]:
# 1 · Settings
from pathlib import Path
ACTION   = 'Train'          # Inspect | Train | Predict   (Predict re-writes the submission from best.pt)
PRESET   = 'MHA-prenorm'    # GQA-prenorm | MHA-prenorm | MHA-postnorm | MLA-prenorm
METHOD   = 'full'           # full | partial | lora | rslora | lorma | ia3 | prefix | kd
RUN_ID   = 'mha-full-win-001'
CONTEXT  = 'window'         # window | sentence
CLASS_WEIGHT = 'sqrt'       # none | sqrt | inverse
EPOCHS, SEED = 8, 17

COURSE_ROOT = Path('/content/drive/MyDrive/legal-llm-course')
PHASE_DIR   = COURSE_ROOT/'Phase_3_RR'
RUNS_ROOT   = COURSE_ROOT/'runs/phase3_rr'
BUNDLE_ZIP  = COURSE_ROOT/'phase2_lner_multimethod_bundle.zip'   # existing Phase 2 bundle: legal_scaffold, training_methods, tokenizer
RR_LIB      = COURSE_ROOT/'rr_lib.py'                            # Phase 3 helper, upload next to the bundle

# adapter / method knobs
RANK, ALPHA = 8, 16                 # lora / rslora / lorma
LORA_TARGETS = ('query', 'value')   # mapped per architecture (MLA -> query_up/value_up)
PARTIAL_LAYERS = (10, 11)
PREFIX_LEN = 16
# knowledge distillation
KD_TEACHER_RUN = RUNS_ROOT/'MHA-prenorm/full/mha-full-win-001'   # finished `full` run used as teacher
KD_STUDENT_LAYERS = (0, 2, 4, 6, 8, 10)                            # 12 -> 6 blocks, copied from Phase 1
KD_TEMPERATURE, KD_ALPHA = 2.0, 0.5

DEFAULT_LR = {  # backbone / adapter / head learning rates
 'full': (5e-5, 0, 1e-3), 'kd': (5e-5, 0, 1e-3), 'partial': (1e-4, 0, 1e-3),
 'lora': (0, 1e-3, 1e-3), 'rslora': (0, 1e-3, 1e-3), 'lorma': (0, 1e-3, 1e-3),
 'ia3': (0, 3e-3, 1e-3), 'prefix': (0, 3e-3, 1e-3)}
assert ACTION in ('Inspect','Train','Predict') and METHOD in DEFAULT_LR and CONTEXT in ('window','sentence')

PRESETS = {
 'GQA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=2,ffn_hidden=2048,context_length=2048,attention='gqa',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MHA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=8,ffn_hidden=2048,context_length=2048,attention='mha',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MHA-postnorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=8,ffn_hidden=2048,context_length=2048,attention='mha',norm_placement='post',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MLA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=2,ffn_hidden=2048,context_length=2048,attention='mla',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02,mla_latent=256,mla_query_rank=256,mla_rope_dim=32)}
# NOTE: presets copied from the Phase 2 notebook (same Phase 1 snapshots, same tokenizer).

In [ ]:
# 2 · Drive, helper bundle, tokenizer, device
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, zipfile, hashlib, json, subprocess, importlib.util, random
LOCAL = Path('/content/phase3-rr')
if LOCAL.exists(): shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as z: z.extractall(LOCAL)
shutil.copy(RR_LIB, LOCAL/'rr_lib.py')
sys.path.insert(0, str(LOCAL))
need = [p for p in ('tokenizers','huggingface_hub','scikit-learn') if importlib.util.find_spec(p.replace('scikit-learn','sklearn')) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])
import torch, rr_lib as R
from tokenizers import Tokenizer
from training_methods import resolve_snapshot_checkpoint
TOKENIZER_SHA256 = '906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab'
tp = LOCAL/'tokenizer/tokenizer.json'
assert hashlib.sha256(tp.read_bytes()).hexdigest() == TOKENIZER_SHA256, 'tokenizer differs from Phase 1/2'
TOK = Tokenizer.from_file(str(tp))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); torch.manual_seed(SEED)
print('device', device, torch.cuda.get_device_name(0) if device.type=='cuda' else '')

In [ ]:
# 3 · Data (downloads Phase_3_RR from the gated HF dataset if not already on Drive)
from google.colab import userdata
try: token = userdata.get('HF_TOKEN')
except Exception: token = None
R.ensure_files(COURSE_ROOT, token)
LABELS = R.load_labels(PHASE_DIR/'Train/label_vocab.json')
L2I = {l:i for i,l in enumerate(LABELS)}
DATA = R.load_all(PHASE_DIR, LABELS)
for s in DATA.values(): R.tokenize_docs(s, TOK, 64)
print(len(LABELS), 'labels:', LABELS)
print(json.dumps(R.summarize(DATA, LABELS), indent=1))
ids = [d['id'] for d in DATA['test']]
print('test ids unique across CL+IT:', len(ids)==len(set(ids)), '| test sentences:', sum(len(d['text']) for d in DATA['test']))
print('example:', DATA['train'][0]['text'][:2], DATA['train'][0]['labels'][:2])

In [ ]:
# 4 · Build run config + model from the Phase 1 snapshot
if CONTEXT == 'window':
    CTX = dict(core_max=12, ctx_left=3, ctx_right=3, budget=512, use_bigru=True, batch_size=8, eval_batch=16)
else:
    CTX = dict(core_max=1, ctx_left=0, ctx_right=0, budget=512, use_bigru=False, batch_size=64, eval_batch=128)
lb, la, lh = DEFAULT_LR[METHOD]
C = dict(method=METHOD, preset=PRESET, run_id=RUN_ID, class_weight=CLASS_WEIGHT, epochs=EPOCHS, seed=SEED,
         lr=dict(backbone=lb, adapter=la, head=lh), partial_layers=PARTIAL_LAYERS,
         kd_temperature=KD_TEMPERATURE, kd_alpha=KD_ALPHA,
         head_kw=dict(hidden=256, use_bigru=CTX['use_bigru'], domain_emb=True, dropout=0.1),
         adapter_kw=dict(rank=RANK, alpha=ALPHA, targets=LORA_TARGETS),
         prefix_len=PREFIX_LEN if METHOD=='prefix' else 0,
         layer_keep=KD_STUDENT_LAYERS if METHOD=='kd' else None, **CTX)
RUN_ROOT = RUNS_ROOT/PRESET/METHOD/RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)

PHASE1_OVERRIDE = {'MHA-prenorm': str(COURSE_ROOT/'runs/mha_pre/run-001/legal-token-stages-v1/run-001-milestone-60762-aa4db459/state.pt')}   # same snapshot Phase 2 used; mha_pre is a Drive shortcut so auto-search misses it

def _scan_snapshots(preset):
    """Diagnostic: list every snapshot.json under runs/ (follows Drive shortcuts) vs the wanted config."""
    import os
    from legal_scaffold.models import ModelConfig
    want = ModelConfig(**PRESETS[preset]).validate().__dict__
    rows = []
    for dp, dn, fn in os.walk(COURSE_ROOT/'runs', followlinks=True):
        if 'snapshot.json' in fn:
            try:
                m = json.loads((Path(dp)/'snapshot.json').read_text())
                ident = m.get('identity', {}); cfg = ident.get('model', ident.get('config')) or {}
                got = ModelConfig(**cfg).validate().__dict__ if cfg else {}
                st = m.get('state_inventory', {}).get('model')
                rows.append((got == want, m.get('stage'), m.get('progress', {}).get('step'), bool(st and (Path(dp)/st).is_file()), dp))
            except Exception as e:
                rows.append((False, 'ERR '+str(e)[:40], None, False, dp))
    return rows

def phase1_checkpoint(preset):
    if preset in PHASE1_OVERRIDE:
        return Path(PHASE1_OVERRIDE[preset]), {'stage': 'override'}
    for kw in (dict(stages=('legal_continued','wikitext_only'), tokenizer_sha256=TOKENIZER_SHA256),
               dict(stages=('legal_continued','wikitext_only'), tokenizer_sha256=None),
               dict(stages=('legal_continued','legal_only','wikitext_only','legal','pretrain'), tokenizer_sha256=None)):
        try:
            return resolve_snapshot_checkpoint(COURSE_ROOT/'runs', PRESETS[preset], **kw)
        except FileNotFoundError:
            pass
    rows = _scan_snapshots(preset)
    print(f'--- No usable Phase 1 snapshot for {preset}. {len(rows)} snapshot.json files found under runs/ ---')
    for r in sorted(rows, key=lambda r: (not r[0], str(r[1])))[:25]:
        print('config_match=%s stage=%s step=%s payload_present=%s  %s' % r)
    raise FileNotFoundError(f'{preset}: no snapshot with matching config + payload; set PHASE1_OVERRIDE[{preset!r}] to its state.pt')


def build(C):
    ck, man = phase1_checkpoint(C['preset'])
    m = R.build_model(PRESETS[C['preset']], ck, len(LABELS), 'full' if C['method']=='kd' else C['method'],
                      head_kw=C['head_kw'], adapter_kw=C['adapter_kw'] if C['method'] not in ('full','kd','partial') else {},
                      prefix_len=C['prefix_len'], layer_keep=C['layer_keep'])
    return m, ck

if ACTION == 'Inspect':
    for p in PRESETS:
        try: print(p, '->', phase1_checkpoint(p)[0])
        except FileNotFoundError as e: print(p, '-> NOT FOUND')
    raise SystemExit('Inspect only.')
model, ckpt = build(C)
print('Phase 1 snapshot:', ckpt)
tr, tot = R.set_trainable(model, METHOD if METHOD!='kd' else 'kd', C['partial_layers'])
print(f'trainable {tr:,} / {tot:,} ({100*tr/tot:.2f}%)   run folder: {RUN_ROOT}')

teacher = None
if METHOD == 'kd':
    tsaved = torch.load(KD_TEACHER_RUN/'best.pt', map_location='cpu', weights_only=False)
    tC = tsaved['C']
    assert tC['method'] in ('full',), 'teacher should be a finished full fine-tune'
    assert tC['core_max']==C['core_max'] and tC['ctx_left']==C['ctx_left'] and tC['ctx_right']==C['ctx_right'], 'teacher/student must use the same windows'
    teacher, _ = build(tC)
    teacher.load_state_dict(tsaved['trainable'], strict=False)
    teacher.to(device).eval()
    for p in teacher.parameters(): p.requires_grad_(False)
    n_t = sum(p.numel() for p in teacher.parameters()); n_s = sum(p.numel() for p in model.parameters())
    assert n_t > n_s, 'teacher must be larger than student'
    print(f'teacher {tC["preset"]} {n_t:,} params (dev F1 {tsaved["dev"]["macro_f1_present"]:.4f}) -> student {PRESET}/6 layers {n_s:,} params')

In [ ]:
# 5 · Train (resumes automatically from last.pt after a Colab disconnect)
if ACTION == 'Train':
    RESULTS, AMP, WCFG = R.train_run(model, DATA, LABELS, RUN_ROOT, C, device, teacher=teacher)
    print('best epoch', RESULTS['best_epoch'], '| dev macro-F1 (present classes)', round(RESULTS['dev']['macro_f1_present'],4),
          '| all-13', round(RESULTS['dev']['macro_f1_all13'],4), '| by domain', {k:round(v['macro_f1_present'],3) for k,v in RESULTS['dev']['by_domain'].items()})
else:
    saved = torch.load(RUN_ROOT/'best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(saved['trainable'], strict=False); model.to(device)
    AMP = torch.bfloat16 if (device.type=='cuda' and torch.cuda.is_bf16_supported()) else torch.float16
    WCFG = dict(core_max=C['core_max'], ctx_left=C['ctx_left'], ctx_right=C['ctx_right'], budget=C['budget'])
    RESULTS = json.loads((RUN_ROOT/'results.json').read_text())

In [ ]:
# 6 · Predict the official test documents and package submission.zip
test_preds, _ = R.predict_docs(model, DATA['test'], WCFG, L2I, device, C['eval_batch'], AMP)
SUB = RUN_ROOT/'submission'
path = R.write_submission(test_preds, DATA['test'], LABELS, SUB)
chk = json.loads(path.read_text())
assert set(chk) == {d['id'] for d in DATA['test']}
assert all(len(chk[d['id']]) == len(d['text']) for d in DATA['test']) and all(l in LABELS for v in chk.values() for l in v)
with zipfile.ZipFile(SUB/'submission.zip') as z: assert z.namelist() == ['prediction.json']
from collections import Counter
print('wrote', SUB/'submission.zip', '| predicted label distribution:', Counter(l for v in chk.values() for l in v).most_common())

In [ ]:
# 7 · Error analysis on dev: per-class F1, hardest confusions, CL vs IT
import pandas as pd
dv = RESULTS['dev']
print(pd.DataFrame(dv['per_class']).T.round(3).sort_values('f1'))
print('\nTop confusions (count, gold -> predicted):')
for n, g, p in R.top_confusions(RESULTS, LABELS, 12): print(f'{n:5d}  {g} -> {p}')
print('\nBy domain:', json.dumps(dv['by_domain'], indent=1))
print('\nTraining curve:'); print(pd.DataFrame(RESULTS['history'])[['epoch','train_loss','dev_macro_f1','minutes']].round(4))

In [ ]:
# 8 · Compare every finished run (for the report's tables)
rows = []
for f in sorted(RUNS_ROOT.rglob('results.json')):
    r = json.loads(f.read_text())
    rows.append({'run': f.parent.name, 'preset': r['preset'], 'method': r['method'],
                 'context': 'window' if r['context']['core_max']>1 else 'sentence',
                 'weights': r['context']['class_weight'],
                 'trainable_M': round(r['trainable_params']/1e6,3), 'total_M': round(r['total_params']/1e6,1),
                 'trainable_%': round(r['trainable_pct'],2), 'best_ep': r['best_epoch'],
                 'dev_F1': round(r['dev']['macro_f1_present'],4), 'F1_CL': round(r['dev']['by_domain'].get('CL',{}).get('macro_f1_present',float('nan')),4),
                 'F1_IT': round(r['dev']['by_domain'].get('IT',{}).get('macro_f1_present',float('nan')),4),
                 'P': round(r['dev']['macro_precision'],4), 'R': round(r['dev']['macro_recall'],4),
                 'min': round(r['train_minutes'] or 0,1), 'peak_GB': r['peak_gpu_gb'] and round(r['peak_gpu_gb'],1)})
if rows: display(pd.DataFrame(rows).sort_values('dev_F1', ascending=False))